In [1]:
# ============================================================
# exp_20260427_060b_cat_depth1_formula_min_gpu_params
# Purpose:
#   既存主力と異なる誤差を作るための、極浅 CatBoost formula/logit 最小素材
#   に対してGPU用にパラメータを変更
#   TE / digit / pairwise / original target mean は使わない
#   S3+057 LR への補助素材化が目的
# ============================================================

In [2]:
from __future__ import annotations

import os
import json
import gc
import random
import warnings
from pathlib import Path
import itertools

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

from catboost import CatBoostClassifier, Pool

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 300)
pd.set_option("display.width", 200)

In [3]:
# ============================================================
# Config
# ============================================================

class CFG:
    EXP_ID = "exp_20260427_060b_cat_depth1_formula_min_gpu_params"
    EXP_NAME = "cat_depth1_formula_min_gpu_params"

    SAVE_DIR = Path(f"/kaggle/working/{EXP_ID}")
    SAVE_DIR.mkdir(parents=True, exist_ok=True)

    TRAIN_PATH = "/kaggle/input/competitions/playground-series-s6e4/train.csv"
    TEST_PATH = "/kaggle/input/competitions/playground-series-s6e4/test.csv"
    SAMPLE_PATH = "/kaggle/input/competitions/playground-series-s6e4/sample_submission.csv"

    TARGET_COL = "Irrigation_Need"
    ID_COL = "id"

    SEED = 42
    N_FOLDS = 5
    NUM_CLASSES = 3

    TARGET_MAP = {"Low": 0, "Medium": 1, "High": 2}
    INV_TARGET_MAP = {0: "Low", 1: "Medium", 2: "High"}

    CAT_PARAMS = {
        "loss_function": "MultiClass",
        "eval_metric": "MultiClass",
        "iterations": 20000,
        "learning_rate": 0.03,
        "depth": 1,
        "l2_leaf_reg": 8.0,
        "random_strength": 1.0,
        "border_count": 254,
        "bootstrap_type": "Bernoulli",
        "subsample": 0.8,
        "random_seed": SEED,
        "early_stopping_rounds": 300,
        "task_type": "GPU",
        "verbose": 500,
        "allow_writing_files": False,
    }

    REF_NPY_PATH = "/kaggle/input/datasets/mizushimatoshihiko/npy-files/"
    REF_OOF = {
        "018": REF_NPY_PATH + "oof_cat_pairwise_te_bias_from_shared_min_proba_biased.npy",
        "046b": REF_NPY_PATH + "oof_xgb_digit_orderedte_min_optuna_biased_refined.npy",
        "025": REF_NPY_PATH + "oof_lgb_digit_te_min_proba_biased.npy",
        "056": REF_NPY_PATH + "oof_cat_pairwise_te_threshold_min_from_family_gpu_default_fix_proba_biased.npy",
        "057": REF_NPY_PATH + "oof_xgb046b_high_interaction_min_biased.npy",
        "060": REF_NPY_PATH + "oof_cat_depth1_formula_min_biased.npy",
    }

In [4]:
# ============================================================
# Utils
# ============================================================

def seed_everything(seed: int):
    random.seed(seed)
    np.random.seed(seed)


def balanced_acc_score_mc(y_true, y_pred):
    if len(y_pred.shape) == 2:
        y_pred = np.argmax(y_pred, axis=1)
    return balanced_accuracy_score(y_true, y_pred)


def save_json(obj, path):
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2)


def safe_logit_like(proba, eps=1e-12):
    return np.log(np.clip(proba, eps, 1.0))


def apply_bias(proba, bias):
    logp = safe_logit_like(proba)
    adjusted = logp + np.array(bias, dtype=np.float32).reshape(1, -1)
    adjusted = adjusted - adjusted.max(axis=1, keepdims=True)
    expv = np.exp(adjusted)
    out = expv / expv.sum(axis=1, keepdims=True)
    return out.astype(np.float32)


def score_bias(proba, y_true, bias):
    adj = apply_bias(proba, bias)
    pred = np.argmax(adj, axis=1)
    return balanced_accuracy_score(y_true, pred)


def run_grid_search(proba, y_true, bias_ranges):
    best_bias = None
    best_score = -1.0

    for b0, b1, b2 in itertools.product(*bias_ranges):
        bias = (b0, b1, b2)
        sc = score_bias(proba, y_true, bias)
        if sc > best_score:
            best_score = sc
            best_bias = bias

    return best_bias, float(best_score)


def mean_raw_corr(a, b):
    vals = []
    for c in range(a.shape[1]):
        vals.append(np.corrcoef(a[:, c], b[:, c])[0, 1])
    return float(np.mean(vals))


def mean_rank_corr(a, b):
    vals = []
    for c in range(a.shape[1]):
        ra = pd.Series(a[:, c]).rank(method="average").to_numpy()
        rb = pd.Series(b[:, c]).rank(method="average").to_numpy()
        vals.append(np.corrcoef(ra, rb)[0, 1])
    return float(np.mean(vals))


seed_everything(CFG.SEED)

In [5]:
# ============================================================
# Load
# ============================================================

train_raw = pd.read_csv(CFG.TRAIN_PATH)
test_raw = pd.read_csv(CFG.TEST_PATH)
sample = pd.read_csv(CFG.SAMPLE_PATH)

train_ids = train_raw[CFG.ID_COL].values
test_ids = test_raw[CFG.ID_COL].values

y = train_raw[CFG.TARGET_COL].map(CFG.TARGET_MAP).astype(int).values

train = train_raw.drop(columns=[CFG.ID_COL, CFG.TARGET_COL]).copy()
test = test_raw.drop(columns=[CFG.ID_COL]).copy()

CATS = [c for c in train.columns if train[c].dtype == "object"]
NUMS = [c for c in train.columns if c not in CATS]

print("train:", train.shape)
print("test :", test.shape)
print("CATS :", CATS)
print("NUMS :", NUMS)

train: (630000, 19)
test : (270000, 19)
CATS : ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
NUMS : ['Soil_pH', 'Soil_Moisture', 'Organic_Carbon', 'Electrical_Conductivity', 'Temperature_C', 'Humidity', 'Rainfall_mm', 'Sunlight_Hours', 'Wind_Speed_kmh', 'Field_Area_hectare', 'Previous_Irrigation_mm']


In [6]:
# ============================================================
# Feature Engineering
# ============================================================

def add_formula_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df["soil_lt_25"] = (df["Soil_Moisture"] < 25).astype("int8")
    df["temp_gt_30"] = (df["Temperature_C"] > 30).astype("int8")
    df["rain_lt_300"] = (df["Rainfall_mm"] < 300).astype("int8")
    df["wind_gt_10"] = (df["Wind_Speed_kmh"] > 10).astype("int8")

    # Mikhail系formula/logit特徴
    tmp = df.copy()

    d = pd.get_dummies(
        tmp[
            [
                "Crop_Growth_Stage",
                "Mulching_Used",
                "soil_lt_25",
                "temp_gt_30",
                "rain_lt_300",
                "wind_gt_10",
            ]
        ],
        drop_first=False,
    )

    z_low = (
        16.3173
        + (-11.0237 * d.get("soil_lt_25", 0))
        + (-5.8559 * d.get("temp_gt_30", 0))
        + (-10.8500 * d.get("rain_lt_300", 0))
        + (-5.8284 * d.get("wind_gt_10", 0))
        + (-5.4155 * d.get("Crop_Growth_Stage_Flowering", 0))
        + (5.5073 * d.get("Crop_Growth_Stage_Harvest", 0))
        + (5.2299 * d.get("Crop_Growth_Stage_Sowing", 0))
        + (-5.4617 * d.get("Crop_Growth_Stage_Vegetative", 0))
        + (-3.0014 * d.get("Mulching_Used_No", 0))
        + (2.8613 * d.get("Mulching_Used_Yes", 0))
    )

    z_med = (
        4.6524
        + (0.3290 * d.get("soil_lt_25", 0))
        + (-0.0204 * d.get("temp_gt_30", 0))
        + (0.1542 * d.get("rain_lt_300", 0))
        + (0.0841 * d.get("wind_gt_10", 0))
        + (0.3586 * d.get("Crop_Growth_Stage_Flowering", 0))
        + (-0.1348 * d.get("Crop_Growth_Stage_Harvest", 0))
        + (-0.3547 * d.get("Crop_Growth_Stage_Sowing", 0))
        + (0.3334 * d.get("Crop_Growth_Stage_Vegetative", 0))
        + (0.1883 * d.get("Mulching_Used_No", 0))
        + (0.0142 * d.get("Mulching_Used_Yes", 0))
    )

    z_high = (
        -20.9697
        + (10.6947 * d.get("soil_lt_25", 0))
        + (5.8763 * d.get("temp_gt_30", 0))
        + (10.6958 * d.get("rain_lt_300", 0))
        + (5.7444 * d.get("wind_gt_10", 0))
        + (5.0569 * d.get("Crop_Growth_Stage_Flowering", 0))
        + (-5.3725 * d.get("Crop_Growth_Stage_Harvest", 0))
        + (-4.8752 * d.get("Crop_Growth_Stage_Sowing", 0))
        + (5.1283 * d.get("Crop_Growth_Stage_Vegetative", 0))
        + (2.8131 * d.get("Mulching_Used_No", 0))
        + (-2.8755 * d.get("Mulching_Used_Yes", 0))
    )

    df["logit_low"] = z_low.astype("float32")
    df["logit_med"] = z_med.astype("float32")
    df["logit_high"] = z_high.astype("float32")

    # 極小High pressure特徴。カテゴリ化はせず、弱い数値として渡す
    df["formula_margin_high_low"] = (df["logit_high"] - df["logit_low"]).astype("float32")
    df["formula_margin_high_med"] = (df["logit_high"] - df["logit_med"]).astype("float32")

    df["moisture_deficit"] = np.maximum(0, 25 - df["Soil_Moisture"]).astype("float32")
    df["rain_deficit"] = np.maximum(0, 300 - df["Rainfall_mm"]).astype("float32")
    df["temp_excess"] = np.maximum(0, df["Temperature_C"] - 30).astype("float32")
    df["wind_excess"] = np.maximum(0, df["Wind_Speed_kmh"] - 10).astype("float32")

    df["high_pressure_score"] = (
        df["soil_lt_25"]
        + df["temp_gt_30"]
        + df["rain_lt_300"]
        + df["wind_gt_10"]
    ).astype("int8")

    return df


def prepare_catboost_frame(train_df: pd.DataFrame, test_df: pd.DataFrame):
    tr = add_formula_features(train_df)
    te = add_formula_features(test_df)

    # 元カテゴリはそのままCatBoostに渡す
    cat_cols = CATS.copy()

    # objectを文字列化
    for c in cat_cols:
        tr[c] = tr[c].astype(str).fillna("__NA__")
        te[c] = te[c].astype(str).fillna("__NA__")

    # 定数列drop
    drop_cols = [c for c in te.columns if te[c].nunique(dropna=False) <= 1]
    if drop_cols:
        print("drop constant:", drop_cols)
        tr = tr.drop(columns=drop_cols)
        te = te.drop(columns=drop_cols)
        cat_cols = [c for c in cat_cols if c not in drop_cols]

    return tr, te, cat_cols


X, X_test, cat_cols = prepare_catboost_frame(train, test)

cat_indices = [X.columns.get_loc(c) for c in cat_cols]

print("X shape:", X.shape)
print("X_test shape:", X_test.shape)
print("cat_cols:", cat_cols)
print("cat_indices:", cat_indices)

X shape: (630000, 33)
X_test shape: (270000, 33)
cat_cols: ['Soil_Type', 'Crop_Type', 'Crop_Growth_Stage', 'Season', 'Irrigation_Type', 'Water_Source', 'Mulching_Used', 'Region']
cat_indices: [0, 10, 11, 12, 13, 14, 16, 18]


In [7]:
# ============================================================
# CV Train
# ============================================================

skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)

oof_raw = np.zeros((len(X), CFG.NUM_CLASSES), dtype=np.float32)
pred_raw = np.zeros((len(X_test), CFG.NUM_CLASSES), dtype=np.float32)

fold_scores = []
best_iterations = []

sample_weights_all = compute_sample_weight(class_weight="balanced", y=y)

for fold, (tr_idx, va_idx) in enumerate(skf.split(X, y), 1):
    print("\n" + "=" * 70)
    print(f"Fold {fold}/{CFG.N_FOLDS}")
    print("=" * 70)

    X_tr = X.iloc[tr_idx].copy()
    y_tr = y[tr_idx]
    X_va = X.iloc[va_idx].copy()
    y_va = y[va_idx]
    w_tr = sample_weights_all[tr_idx]

    train_pool = Pool(
        X_tr,
        label=y_tr,
        cat_features=cat_indices,
        weight=w_tr,
    )

    valid_pool = Pool(
        X_va,
        label=y_va,
        cat_features=cat_indices,
    )

    test_pool = Pool(
        X_test,
        cat_features=cat_indices,
    )

    params = CFG.CAT_PARAMS.copy()

    model = CatBoostClassifier(**params)
    model.fit(
        train_pool,
        eval_set=valid_pool,
        use_best_model=True,
    )

    va_pred = model.predict_proba(valid_pool).astype(np.float32)
    te_pred = model.predict_proba(test_pool).astype(np.float32)

    oof_raw[va_idx] = va_pred
    pred_raw += te_pred / CFG.N_FOLDS

    fold_score = balanced_acc_score_mc(y_va, va_pred)
    fold_scores.append(float(fold_score))

    best_it = model.get_best_iteration()
    best_iterations.append(int(best_it) if best_it is not None else None)

    print(f"fold BA: {fold_score:.9f}")
    print(f"best_iteration: {best_it}")

    del X_tr, X_va, train_pool, valid_pool, test_pool, model, va_pred, te_pred
    gc.collect()


raw_cv = balanced_acc_score_mc(y, oof_raw)

print("\nraw_cv:", raw_cv)
print("fold_scores:", fold_scores)
print("mean_fold:", float(np.mean(fold_scores)))
print("best_iterations:", best_iterations)


Fold 1/5
0:	learn: 1.0702041	test: 1.0597169	best: 1.0597169 (0)	total: 6.59s	remaining: 1d 12h 37m 43s
500:	learn: 0.1087300	test: 0.0899249	best: 0.0899249 (500)	total: 9.45s	remaining: 6m 7s
1000:	learn: 0.1008748	test: 0.0837793	best: 0.0837793 (1000)	total: 12.3s	remaining: 3m 53s
1500:	learn: 0.0975192	test: 0.0815355	best: 0.0815355 (1500)	total: 15.1s	remaining: 3m 6s
2000:	learn: 0.0953687	test: 0.0801922	best: 0.0801922 (2000)	total: 17.9s	remaining: 2m 40s
2500:	learn: 0.0937809	test: 0.0793086	best: 0.0793086 (2500)	total: 20.7s	remaining: 2m 24s
3000:	learn: 0.0925558	test: 0.0786333	best: 0.0786333 (3000)	total: 23.5s	remaining: 2m 13s
3500:	learn: 0.0916213	test: 0.0781197	best: 0.0781197 (3500)	total: 26.3s	remaining: 2m 3s
4000:	learn: 0.0907176	test: 0.0775344	best: 0.0775344 (4000)	total: 29.1s	remaining: 1m 56s
4500:	learn: 0.0900053	test: 0.0771208	best: 0.0771208 (4500)	total: 31.9s	remaining: 1m 49s
5000:	learn: 0.0893474	test: 0.0767341	best: 0.0767341 (5000)	t

In [8]:
# ============================================================
# Bias refine
# 046b準拠の3-stage grid
# ============================================================

coarse_vals = np.arange(-1.0, 1.01, 0.1)
best_bias_1, best_score_1 = run_grid_search(
    oof_raw,
    y,
    [coarse_vals, coarse_vals, coarse_vals],
)

print("stage1 best_bias:", best_bias_1, "score:", best_score_1)

b0, b1, b2 = best_bias_1
local_vals_0 = np.arange(b0 - 0.10, b0 + 0.1001, 0.02)
local_vals_1 = np.arange(b1 - 0.10, b1 + 0.1001, 0.02)
local_vals_2 = np.arange(b2 - 0.10, b2 + 0.1001, 0.02)

best_bias_2, best_score_2 = run_grid_search(
    oof_raw,
    y,
    [local_vals_0, local_vals_1, local_vals_2],
)

print("stage2 best_bias:", best_bias_2, "score:", best_score_2)

b0, b1, b2 = best_bias_2
ultra_vals_0 = np.arange(b0 - 0.02, b0 + 0.0201, 0.005)
ultra_vals_1 = np.arange(b1 - 0.02, b1 + 0.0201, 0.005)
ultra_vals_2 = np.arange(b2 - 0.02, b2 + 0.0201, 0.005)

best_bias_3, best_score_3 = run_grid_search(
    oof_raw,
    y,
    [ultra_vals_0, ultra_vals_1, ultra_vals_2],
)

print("stage3 best_bias:", best_bias_3, "score:", best_score_3)

final_bias = best_bias_3
final_cv = best_score_3

oof_biased = apply_bias(oof_raw, final_bias)
pred_biased = apply_bias(pred_raw, final_bias)

print("raw_cv:", raw_cv)
print("final_cv:", final_cv)
print("final_bias:", final_bias)

stage1 best_bias: (np.float64(-1.0), np.float64(-0.7000000000000001), np.float64(-0.40000000000000013)) score: 0.9716334409414525
stage2 best_bias: (np.float64(-1.1), np.float64(-0.78), np.float64(-0.5000000000000001)) score: 0.9716779701333916
stage3 best_bias: (np.float64(-1.12), np.float64(-0.8), np.float64(-0.5200000000000001)) score: 0.9716779701333916
raw_cv: 0.9712866029568512
final_cv: 0.9716779701333916
final_bias: (np.float64(-1.12), np.float64(-0.8), np.float64(-0.5200000000000001))


In [9]:
# ============================================================
# Save artifacts
# ============================================================

np.save(CFG.SAVE_DIR / "oof_cat_depth1_formula_min_gpu_params_raw.npy", oof_raw)
np.save(CFG.SAVE_DIR / "pred_cat_depth1_formula_min_gpu_params_raw.npy", pred_raw)

np.save(CFG.SAVE_DIR / "oof_cat_depth1_formula_min_gpu_params_biased.npy", oof_biased)
np.save(CFG.SAVE_DIR / "pred_cat_depth1_formula_min_gpu_params_biased.npy", pred_biased)

np.save(CFG.SAVE_DIR / "best_class_bias_refined.npy", np.array(final_bias, dtype=np.float32))

submission = sample.copy()
submission[CFG.TARGET_COL] = pd.Series(np.argmax(pred_biased, axis=1)).map(CFG.INV_TARGET_MAP)
submission.to_csv(CFG.SAVE_DIR / f"submission_{CFG.EXP_NAME}.csv", index=False)

cv_result = {
    "exp_id": CFG.EXP_ID,
    "exp_name": CFG.EXP_NAME,
    "base_parent": "exp_20260427_060_cat_depth1_formula_min",
    "raw_cv": float(raw_cv),
    "refined_biased_cv": float(final_cv),
    "best_bias": [float(x) for x in final_bias],
    "fold_scores": fold_scores,
    "best_iterations": best_iterations,
    "model": "CatBoostClassifier depth=1 formula/logit minimal gpu_params",
    "cat_params": CFG.CAT_PARAMS,
    "features": {
        "used": [
            "raw numerical",
            "raw categorical",
            "formula logit_low/logit_med/logit_high",
            "High flags",
            "formula margins",
            "deficit/excess numeric features",
        ],
        "not_used": [
            "TargetEncoder",
            "digit features",
            "pairwise categorical combinations",
            "original target mean",
            "frequency encoding",
        ],
    },
    "purpose": "low-correlation auxiliary material for S3+057 LR",
    "bias_search": {
        "stage1": {"range": [-1.0, 1.0], "step": 0.1},
        "stage2": {"center": [float(x) for x in best_bias_1], "width": 0.10, "step": 0.02},
        "stage3": {"center": [float(x) for x in best_bias_2], "width": 0.02, "step": 0.005},
    },
}

save_json(cv_result, CFG.SAVE_DIR / "cv_result.json")

print("saved to:", CFG.SAVE_DIR)

saved to: /kaggle/working/exp_20260427_060b_cat_depth1_formula_min_gpu_params


In [10]:
# ============================================================
# Correlation vs existing OOF
# ============================================================

corr_rows = []

for name, path in CFG.REF_OOF.items():
    if os.path.exists(path):
        ref = np.load(path)
        corr_rows.append({
            "ref": name,
            "raw_corr_mean": mean_raw_corr(oof_biased, ref),
            "rank_corr_mean": mean_rank_corr(oof_biased, ref),
        })
    else:
        print("missing ref:", name, path)

corr_df = pd.DataFrame(corr_rows).sort_values("rank_corr_mean", ascending=False)
corr_df.to_csv(CFG.SAVE_DIR / "corr_vs_existing_oof.csv", index=False)

display(corr_df)

print("Done.")

,ref,raw_corr_mean,rank_corr_mean
5,060,0.999654,0.998789
1,046b,0.975484,0.943468
4,057,0.974456,0.940373
3,056,0.974031,0.933959
2,025,0.975416,0.933644
0,018,0.964635,0.906574


Done.
